## Building an NVFlare Job

In this section, we'll learn how to configure and run a federated learning job using NVFlare's simulation mode.

### Core Concepts

An NVFlare **job** consists of three main components:
1. **Controller** (Server-side): Orchestrates the federated learning workflow
2. **Persistor** (Server-side): Manages model storage and loading
3. **Executor** (Client-side): Runs your training script

Let's build a job step by step.

### Step 1: Create the Job

The `FedJob` object is the container for your entire federated learning configuration.

**Key Parameters:**
- `name`: Identifier for this job (useful when running multiple experiments)
- `min_clients`: Minimum number of clients that must connect before training starts

In [ ]:
from nvflare import FedJob

# Job configuration
job_name = "NVFlare_Simulation_Demo"


# Create the federated job
job = FedJob(
    name=job_name
)

### Step 2: Configure the Controller

The **controller** implements the federated learning algorithm on the server side.

**FedAvg Controller:**
- Implements the Federated Averaging algorithm we manually coded in the previous tutorial
- Receives model updates from clients
- Aggregates weights (weighted average based on dataset size)
- Distributes the updated global model back to clients

**Parameters:**
- `num_clients`: How many clients to expect per round
- `num_rounds`: How many federated learning rounds to execute

In [ ]:
from nvflare.app_common.workflows.fedavg import FedAvg

# Configure the FedAvg controller
num_clients = 3
num_rounds = 10

# Create the FedAvg controller
controller = FedAvg(
    num_clients=num_clients,
    num_rounds=num_rounds
)

# Add controller to the server
job.to_server(controller)

### Step 3: Configure the Model Persistor

The **persistor** manages model storage on the server side.

**PTFileModelPersistor:**
- **Initial Model**: Loads your model architecture to get the initial weights
- **Saving**: Persists the global model after each round
- **Loading**: Can load a pre-trained model to continue training

**Why is this needed?**
- Preserves the global model between rounds
- Enables checkpointing (resume training if interrupted)
- Stores the final trained model for deployment

In [ ]:
from nvflare.app_opt.pt.file_model_persistor import PTFileModelPersistor
from pathlib import Path

import os
import sys
sys.path.insert(0, os.path.abspath(".."))
from src.utils.train_util import SimpleClassifier

## Configure model persistor
# This is where the global model weights will be saved to disk at the end of each round
persistor_global_file_name = Path(f"{Path.cwd()}/../output/global_model.pt")
# Ensure the output directory exists
persistor_global_file_name.parent.mkdir(parents=True, exist_ok=True)
# This is the initial architecture the persistor will use to store weights
model_object = SimpleClassifier()

## Create code for model output

# Create the persistor
persistor = PTFileModelPersistor(
    model=model_object,
    global_model_file_name=str(persistor_global_file_name),
    allow_numpy_conversion=False
)

# Add persistor to the server
job.to_server(persistor, id="persistor")

### Step 4: Configure the Client Executor

The **executor** runs on each client and executes your training script.

**ScriptRunner:**
- Runs your `train.py` script in each client's environment
- Handles communication between your script and the NVFlare framework
- Manages the `flare.receive()` and `flare.send()` operations

**Key Parameters:**
- `script`: Path to your training script

In [ ]:
from nvflare.job_config.script_runner import ScriptRunner

# Configure the script runner (client executor)
script_location = os.path.abspath("../src/train.py")

runner = ScriptRunner(
            script=script_location,
            server_expected_format="pytorch"
        )

# Add executor to all clients
job.to_clients(runner, tasks=["train"])

### Step 5: Export the Job Configuration

Before we can run the simulation, we need to **export** the job. This step is critical for two reasons:

1. **Creates the required directory structure** for NVFlare to execute the job
2. **Generates production-ready configuration files** that can be deployed to real federated learning infrastructure

Let's export the job and examine what gets created.

In [ ]:
job.export_job("../simulation")

### Understanding the Export Structure

You may have noticed a new folder in your current working directory called `simulation/`. 


Within that subdirectory the `export_job()` method creates a standardized directory structure:
```
job_name/
├── app/
│   ├── config/
│   │   └── config_fed_server.json    # Server configuration
│   │   └── config_fed_client.json    # Client configuration
│   └── custom/
│       └── train.py                  # Your training script
└── meta.json                         # Job metadata
```
We wont go into detail in this tutorial, but these files and structure are identical to what will be required for production deployement of federated learning jobs. These files will be covered in the following tutorial.

### Why This Matters

**For Simulation:**
- NVFlare's simulator reads these configuration files to set up the federated environment
- Each simulated client gets its own copy of the configuration

**For Production Deployment:**
- ✅ These files are **exactly what you need** to deploy to a real federated learning infrastructure
- ✅ Only minor changes needed to files to update pathing

### Step 6: Run the Federated Learning Simulation

Now that we've configured and exported our job, we're ready to run the simulation!

NVFlare's simulator will:
1. Create virtual clients based on our configuration
2. Start a virtual server to orchestrate training
3. Execute the federated learning workflow locally
4. Save results and logs for analysis

In [ ]:
job.simulator_run("../simulation/workdir", n_clients=num_clients, log_config="../src/utils/log_config.json")

### Understanding the Output

As the simulation runs, you'll see logs showing the progress of federated learning with outputs indicating:
- **Round X/Y Start**: The server has initiated round X of Y total rounds
- **Site-X Global Model Local Accuracy**: How well the global model performs on Site-X's local test data
- **Cross-Site Global Accuracy**: The aggregated accuracy across all participating sites 

**Key Observations:**
All three clients benefited and produced a pretty strong global model. You may also notice there are model weights saved in `output/`. This is the global model from the federated training.

---

### Optionally
In step 2, change `num_clients` to `1` , restart the kernel, and rerun the experiment. This will simulate a local training on a single client and use the same dataset client 1 used originally.

Running this experiment can help solidify the impact federated learning has on addressing real world data challenges.

---

### 🎓 What You've Learned
**NVFlare Simulation:**
- ✅ Configured a complete FL job programmatically
- ✅ Exported configuration files
- ✅ Ran a federated learning simulation locally

---

### Next Steps

In the next tutorial, you'll learn how to deploy these same configuration files to a real federated learning platform (Rhino FCP) and run distributed training across actual remote sites using real data and a real model.

**Continue to:** `Tutorial 3 - Deploying_Federated_Learning`